#### Case Study по модулю SQL

#### Исполнитель: Abdurahmonova Marjona
#### Дата исполнения: 1.04.25

In [115]:
import pandas as pd
import psycopg2

In [116]:
from db_connection import create_connection

In [117]:
conn = create_connection()

In [22]:
# Загрузка данных из файла Excel
customers = pd.read_excel('adventure_works.xlsx')
sales_data = pd.read_excel('adventure_works.xlsx')

# Запись данных в таблицу
customers.to_sql('customers', con=engine, schema='adv_works', if_exists='replace', index=False)
sales_data.to_sql('sales', schema='adv_works', con=engine, if_exists='replace', index=False)

484

In [118]:
from db_connection import create_connection

conn = create_connection()
if conn:
    cursor = conn.cursor()
    cursor.execute("SELECT 1;")
    print(cursor.fetchone())
else:
    print("Не удалось подключиться к базе данных.")

(1,)


## Блок 1: Подготовка таблицы
### Описание задачи:
Создание схемы и таблиц для анализа данных.


In [68]:
# Создание схемы и таблицы
create_script = """
CREATE SCHEMA adv_works;

CREATE TABLE adv_works.customers (
    "CustomerKey" INT PRIMARY KEY,
    "CustomerName" VARCHAR(100),
    "YearlyIncome" FLOAT,
    "Occupation" VARCHAR(50),
    "MaritalStatus" CHAR(1),
    "HasChildren" BOOLEAN,
    "GeographyKey" INT,
    "DateFirstPurchase" DATE,
    "NumberChildrenAtHome" INT
);

ALTER TABLE adv_works.customers ADD COLUMN "SalesAmount" NUMERIC;

UPDATE adv_works.customers
SET "SalesAmount" = "YearlyIncome" * 0.3;  -- Предполагается, что клиенты тратят 30% своего дохода.
"""
try:
    cursor.execute(create_script)
    conn.commit()
    print("Скрипт выполнен успешно!")
except Exception as e:
    print("Ошибка при выполнении скрипта:", e)
    conn.rollback()

Ошибка при выполнении скрипта: ОШИБКА:  схема "adv_works" уже существует



In [87]:
update_query = """
UPDATE adv_works.customers
SET salesamount = "YearlyIncome" * 0.3
WHERE salesamount IS NULL;
"""

try:
    cursor.execute(update_query)
    conn.commit()
    print("Значения столбца 'salesamount' обновлены.")
except Exception as e:
    print("Ошибка при выполнении скрипта:", e)
    conn.rollback()


Значения столбца 'salesamount' обновлены.


Выводы:
Таблица customers успешно создана.

### Блок 2: Аналитические задачи
Секция 1: Анализ клиентов

Задача: Сегментация по доходу

In [119]:
query = """
SELECT 
    "Occupation",
    COUNT(*) AS "NumberOfCustomers",
    AVG("YearlyIncome") AS "AvgIncome"
FROM adv_works.customers
GROUP BY "Occupation";
"""
df = pd.read_sql(query, con=engine)
print(df)


       Occupation  NumberOfCustomers     AvgIncome
0      Management               3075  92325.203252
1        Clerical               2928  30710.382514
2          Manual               2384  16451.342282
3  Skilled Manual               4577  51715.097225
4    Professional               5520  74184.782609


Семейный профиль

In [103]:
query = """
SELECT 
    CASE 
        WHEN "NumberChildrenAtHome" > 0 THEN 1
        ELSE 0
    END AS "HasChildren",
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM adv_works.customers), 2) AS "PctOfCustomerBase"
FROM adv_works.customers
GROUP BY 
    CASE 
        WHEN "NumberChildrenAtHome" > 0 THEN 1
        ELSE 0
    END;
"""
df = pd.read_sql(query, con=engine)
print(df)

   HasChildren  PctOfCustomerBase
0            0              60.14
1            1              39.86


Высокодоходные клиенты

In [105]:
query = """
SELECT 
    "CustomerKey",
    "Name" AS customer_name,
    SUM(salesamount) AS total_purchase
FROM adv_works.customers
GROUP BY "CustomerKey", "Name"
ORDER BY total_purchase DESC
LIMIT 10;
"""
# df = pd.read_sql(query, conn)
# print(df)

df = pd.read_sql(query, con=engine)
print(df)

   CustomerKey       customer_name  total_purchase
0        20558     Brianna Stewart         51000.0
1        21822        Gregory Yuan         51000.0
2        13408  Katherine Gonzales         51000.0
3        15699  Jonathon Hernandez         51000.0
4        12658           Joy Gomez         51000.0
5        17182   Jessica Patterson         51000.0
6        22822      Devin Anderson         51000.0
7        28880       Felicia Moyer         51000.0
8        16643     Fernando Wilson         51000.0
9        16869   Vanessa Patterson         51000.0


Влияние семейного положения

In [106]:
query = """ 
SELECT 
    EXTRACT(YEAR FROM "DateFirstPurchase") AS "Year",
    "MaritalStatus",
    AVG(salesamount) AS "AvgSalesAmount"
FROM adv_works.customers
GROUP BY "Year", "MaritalStatus"
ORDER BY "Year", "MaritalStatus";
SELECT 
    "Year",
    MAX("AvgSalesAmount") - MIN("AvgSalesAmount") AS "IncomeDifference"
FROM (
    SELECT 
        EXTRACT(YEAR FROM "DateFirstPurchase") AS "Year",
        "MaritalStatus",
        AVG(salesamount) AS "AvgSalesAmount"
    FROM adv_works.customers
    GROUP BY "Year", "MaritalStatus"
) AS "Subquery"
GROUP BY "Year";
"""
df = pd.read_sql(query, con=engine)
print(df)

     Year  IncomeDifference
0  2003.0       2043.022021
1  2002.0       1639.402596
2  2001.0        835.664336
3  2004.0       2414.898227


Выводы:
Профессии с самым высоким средним доходом — [результаты].

#### Секция 2: Анализ продаж

Задача: Ежемесячные продажи

In [107]:
query = """
SELECT 
    EXTRACT(YEAR FROM "DateFirstPurchase") AS "Year",
    EXTRACT(MONTH FROM "DateFirstPurchase") AS "MonthKey",
    TO_CHAR("DateFirstPurchase", 'Month') AS "MonthName",
    COUNT(*) AS "SalesCount",
    SUM(salesamount) AS "SalesAmount"
FROM adv_works.customers
WHERE EXTRACT(YEAR FROM "DateFirstPurchase") IN (2003, 2004)
GROUP BY 
    EXTRACT(YEAR FROM "DateFirstPurchase"),
    EXTRACT(MONTH FROM "DateFirstPurchase"),
    TO_CHAR("DateFirstPurchase", 'Month')
ORDER BY "Year", "MonthKey";
"""
df = pd.read_sql(query, con=engine)
print(df)

      Year  MonthKey  MonthName  SalesCount  SalesAmount
0   2003.0       1.0  January           244    4413000.0
1   2003.0       2.0  February          272    4497000.0
2   2003.0       3.0  March             272    5067000.0
3   2003.0       4.0  April             294    3966000.0
4   2003.0       5.0  May               335    6366000.0
5   2003.0       6.0  June              321    6678000.0
6   2003.0       7.0  July              202    3642000.0
7   2003.0       8.0  August           1210   20865000.0
8   2003.0       9.0  September        1112   18786000.0
9   2003.0      10.0  October          1132   19239000.0
10  2003.0      11.0  November         1094   18621000.0
11  2003.0      12.0  December         1210   20061000.0
12  2004.0       1.0  January          1039   17178000.0
13  2004.0       2.0  February         1013   16965000.0
14  2004.0       3.0  March            1056   16806000.0
15  2004.0       4.0  April            1088   18330000.0
16  2004.0       5.0  May      

Продажи по регионам

In [108]:
query = """ 
SELECT 
    "GeographyKey" AS "Region",
    COUNT(*) AS "SalesCount",
    SUM(salesamount) AS "SalesAmount"
FROM adv_works.customers
GROUP BY "GeographyKey"
ORDER BY "SalesAmount" DESC;
"""
df = pd.read_sql(query, con=engine)
print(df)

     Region  SalesCount  SalesAmount
0       311         212    4155000.0
1       307         206    4056000.0
2       299         200    3858000.0
3       609         210    3828000.0
4       302         198    3738000.0
..      ...         ...          ...
331     573           1       9000.0
332     608           1       9000.0
333     402           1       9000.0
334     431           1       9000.0
335     524           1       6000.0

[336 rows x 3 columns]


Выводы:
Наибольшие продажи наблюдаются в [месяц].

#### Секция 3: Анализ продуктов

Задача: Доля продаж

In [109]:
query = """
SELECT 
    EXTRACT(YEAR FROM "DateFirstPurchase") AS "Year",
    "GeographyKey" AS "Region",
    SUM(salesamount) AS "SalesAmount",
    ROUND(SUM(salesamount) * 100.0 / SUM(SUM(salesamount)) OVER (PARTITION BY EXTRACT(YEAR FROM "DateFirstPurchase")), 2) AS "PctOfTotalSales"
FROM adv_works.customers
GROUP BY 
    EXTRACT(YEAR FROM "DateFirstPurchase"),
    "GeographyKey"
ORDER BY "Year", "PctOfTotalSales" DESC;
"""
df = pd.read_sql(query, con=engine)
print(df)


        Year  Region  SalesAmount  PctOfTotalSales
0     2001.0       4     387000.0             2.15
1     2001.0      13     384000.0             2.14
2     2001.0      40     306000.0             1.70
3     2001.0     301     294000.0             1.64
4     2001.0      30     288000.0             1.60
...      ...     ...          ...              ...
1147  2004.0     573       9000.0             0.01
1148  2004.0     488      12000.0             0.01
1149  2004.0     653      15000.0             0.01
1150  2004.0     513      12000.0             0.01
1151  2004.0     125       6000.0             0.01

[1152 rows x 4 columns]


2. Самые продаваемые продукты

In [110]:
query = """ 
SELECT 
    "CustomerKey" AS "ProductKey",
    "Name" AS "ProductName",
    'Customer' AS "EnglishProductCategoryName",
    SUM(salesamount) AS "SalesAmount"
FROM adv_works.customers
GROUP BY "CustomerKey", "Name"
ORDER BY "SalesAmount" DESC
LIMIT 5;
"""
df = pd.read_sql(query, con=engine)
print(df)

   ProductKey         ProductName EnglishProductCategoryName  SalesAmount
0       15699  Jonathon Hernandez                   Customer      51000.0
1       12658           Joy Gomez                   Customer      51000.0
2       17182   Jessica Patterson                   Customer      51000.0
3       16643     Fernando Wilson                   Customer      51000.0
4       22822      Devin Anderson                   Customer      51000.0


3. Маржа от продаж

In [111]:
query = """ 
SELECT 
    EXTRACT(YEAR FROM "DateFirstPurchase") AS "Year",
    EXTRACT(MONTH FROM "DateFirstPurchase") AS "MonthKey",
    TO_CHAR("DateFirstPurchase", 'Month') AS "MonthName",
    "CustomerKey" AS "ProductKey",
    "Name" AS "ProductName",
    SUM(salesamount) AS "SalesAmount",
    SUM(salesamount) AS "Margin", -- Маржа = salesamount (нет дополнительных данных)
    ROUND(SUM(salesamount) * 100.0 / SUM(SUM(salesamount)) OVER (PARTITION BY EXTRACT(YEAR FROM "DateFirstPurchase")), 2) AS "MarginPct"
FROM adv_works.customers
GROUP BY 
    EXTRACT(YEAR FROM "DateFirstPurchase"),
    EXTRACT(MONTH FROM "DateFirstPurchase"),
    TO_CHAR("DateFirstPurchase", 'Month'),
    "CustomerKey",
    "Name"
ORDER BY "Year", "MonthKey", "MarginPct" DESC;
"""
df = pd.read_sql(query, con=engine)
print(df)

         Year  MonthKey  MonthName  ProductKey      ProductName  SalesAmount  \
0      2001.0       7.0  July            19942  Armando Navarro      51000.0   
1      2001.0       7.0  July            13591   Latasha Alonso      51000.0   
2      2001.0       7.0  July            27663    Aaron Collins      51000.0   
3      2001.0       7.0  July            13590        Louis Xie      51000.0   
4      2001.0       7.0  July            19941     Cedric Liang      48000.0   
...       ...       ...        ...         ...              ...          ...   
18479  2004.0       7.0  July            20136   Bonnie Kennedy       3000.0   
18480  2004.0       7.0  July            20144        Roger Lin       3000.0   
18481  2004.0       7.0  July            12233       Jerry Yuan       3000.0   
18482  2004.0       7.0  July            20168   Jessie Serrano       3000.0   
18483  2004.0       7.0  July            26361     Ebony Suarez       3000.0   

        Margin  MarginPct  
0      5100

Вывод:
Регион с наибольшей долей продаж — [результаты].

#### Секция 4: Анализ трендов

Задача: Квартальный рост:

квартальная динамика продаж по регионам (GeographyKey)

In [112]:
query = """
WITH QuarterlySales AS (
    SELECT 
        EXTRACT(YEAR FROM "DateFirstPurchase") AS year,
        EXTRACT(QUARTER FROM "DateFirstPurchase") AS quarter_id,
        "GeographyKey" AS product_category_key, -- Используем регионы как категории
        'Region' AS english_product_category_name,
        SUM(salesamount) AS quarter_sales_amount
    FROM adv_works.customers
    GROUP BY year, quarter_id, "GeographyKey"
),
QuarterlyGrowth AS (
    SELECT 
        year,
        quarter_id,
        product_category_key,
        english_product_category_name,
        quarter_sales_amount,
        LAG(quarter_sales_amount) OVER (PARTITION BY product_category_key ORDER BY year, quarter_id) AS prev_quarter_sales
    FROM QuarterlySales
)
SELECT 
    year,
    quarter_id,
    product_category_key,
    english_product_category_name,
    quarter_sales_amount,
    ROUND((quarter_sales_amount - prev_quarter_sales) * 100.0 / NULLIF(prev_quarter_sales, 0), 2) AS quarter_over_quarter_growth_pct
FROM QuarterlyGrowth
ORDER BY year, quarter_id, product_category_key;

--Этот запрос вычисляет квартальные суммы продаж и их рост по регионам (заменяя категории продуктов).
""" 
df = pd.read_sql(query, con=engine)
print(df)

        year  quarter_id  product_category_key english_product_category_name  \
0     2001.0         3.0                     2                        Region   
1     2001.0         3.0                     3                        Region   
2     2001.0         3.0                     4                        Region   
3     2001.0         3.0                     5                        Region   
4     2001.0         3.0                     6                        Region   
...      ...         ...                   ...                           ...   
3176  2004.0         3.0                   638                        Region   
3177  2004.0         3.0                   641                        Region   
3178  2004.0         3.0                   642                        Region   
3179  2004.0         3.0                   644                        Region   
3180  2004.0         3.0                   648                        Region   

      quarter_sales_amount  quarter_ove

Сравнение будних и выходных дней

In [113]:
query = """
SELECT 
    EXTRACT(YEAR FROM "DateFirstPurchase") AS year,
    TO_CHAR("DateFirstPurchase", 'Day') AS day_name,
    CASE 
        WHEN TO_CHAR("DateFirstPurchase", 'D') IN ('1', '7') THEN 1 -- Суббота, воскресенье
        ELSE 0
    END AS is_weekend,
    SUM(salesamount) AS sales_amount
FROM adv_works.customers
GROUP BY year, day_name, is_weekend
ORDER BY year, is_weekend DESC, sales_amount DESC;
"""
df = pd.read_sql(query, con=engine)
print(df)

      year   day_name  is_weekend  sales_amount
0   2001.0  Sunday              1     2979000.0
1   2001.0  Saturday            1     2673000.0
2   2001.0  Monday              0     2724000.0
3   2001.0  Friday              0     2490000.0
4   2001.0  Thursday            0     2484000.0
5   2001.0  Wednesday           0     2334000.0
6   2001.0  Tuesday             0     2283000.0
7   2002.0  Saturday            1     7092000.0
8   2002.0  Sunday              1     6885000.0
9   2002.0  Wednesday           0     7527000.0
10  2002.0  Thursday            0     7074000.0
11  2002.0  Tuesday             0     7041000.0
12  2002.0  Monday              0     6915000.0
13  2002.0  Friday              0     6795000.0
14  2003.0  Saturday            1    18831000.0
15  2003.0  Sunday              1    18555000.0
16  2003.0  Friday              0    19485000.0
17  2003.0  Wednesday           0    19392000.0
18  2003.0  Tuesday             0    19251000.0
19  2003.0  Monday              0    183

Вывод:
Продажи выше в [будние/выходные] дни.